In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# 1. Load Datasets[cite: 112, 113]
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# 2. Data Preprocessing & Feature Engineering
data = [train_df, test_df]
deck = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}

for dataset in data:
    # Handle Cabin / Deck[cite: 120]
    dataset['Cabin'] = dataset['Cabin'].fillna("U0")
    dataset['Deck'] = dataset['Cabin'].map(lambda x: re.compile("([a-zA-Z]+)").search(x).group())
    dataset['Deck'] = dataset['Deck'].map(deck).fillna(0).astype(int)
    
    # Handle Age missing values using random numbers based on mean/std[cite: 120]
    mean, std, is_null = dataset["Age"].mean(), dataset["Age"].std(), dataset["Age"].isnull().sum()
    rand_age = np.random.randint(mean - std, mean + std, size=is_null)
    age_slice = dataset["Age"].copy()
    age_slice[np.isnan(age_slice)] = rand_age
    dataset["Age"] = age_slice.astype(int)
    
    # Age Grouping[cite: 123]
    dataset.loc[dataset['Age'] <= 11, 'Age'] = 0
    dataset.loc[(dataset['Age'] > 11) & (dataset['Age'] <= 18), 'Age'] = 1
    dataset.loc[(dataset['Age'] > 18) & (dataset['Age'] <= 22), 'Age'] = 2
    dataset.loc[(dataset['Age'] > 22) & (dataset['Age'] <= 27), 'Age'] = 3
    dataset.loc[(dataset['Age'] > 27) & (dataset['Age'] <= 33), 'Age'] = 4
    dataset.loc[(dataset['Age'] > 33) & (dataset['Age'] <= 40), 'Age'] = 5
    dataset.loc[(dataset['Age'] > 40) & (dataset['Age'] <= 66), 'Age'] = 6
    dataset.loc[dataset['Age'] > 66, 'Age'] = 6

    # Handle Embarked and Fare missing values[cite: 121]
    dataset['Embarked'] = dataset['Embarked'].fillna('S')
    dataset['Fare'] = dataset['Fare'].fillna(0).astype(int)
    
    # Fare Grouping[cite: 124]
    dataset.loc[dataset['Fare'] <= 7.91, 'Fare'] = 0
    dataset.loc[(dataset['Fare'] > 7.91) & (dataset['Fare'] <= 14.454), 'Fare'] = 1
    dataset.loc[(dataset['Fare'] > 14.454) & (dataset['Fare'] <= 31), 'Fare'] = 2
    dataset.loc[(dataset['Fare'] > 31) & (dataset['Fare'] <= 99), 'Fare'] = 3
    dataset.loc[(dataset['Fare'] > 99) & (dataset['Fare'] <= 250), 'Fare'] = 4
    dataset.loc[dataset['Fare'] > 250, 'Fare'] = 5

    # Relatives and Alone features[cite: 119]
    dataset['relatives'] = dataset['SibSp'] + dataset['Parch']
    dataset['not_alone'] = (dataset['relatives'] == 0).astype(int)

    # Extract Titles from Name[cite: 122]
    titles = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
    dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr',
                                                 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace(['Mlle', 'Ms'], 'Miss').replace('Mme', 'Mrs')
    dataset['Title'] = dataset['Title'].map(titles).fillna(0)

    # Map categorical features to numbers[cite: 122]
    dataset['Sex'] = dataset['Sex'].map({"male": 0, "female": 1})
    dataset['Embarked'] = dataset['Embarked'].map({"S": 0, "C": 1, "Q": 2})

    # Interaction features[cite: 124]
    dataset['Age_Class'] = dataset['Age'] * dataset['Pclass']
    dataset['Fare_Per_Person'] = (dataset['Fare'] / (dataset['relatives'] + 1)).astype(int)

# Drop redundant columns[cite: 131]
train_df = train_df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin', 'not_alone', 'Parch'], axis=1)
test_df = test_df.drop(['Name', 'Ticket', 'Cabin', 'not_alone', 'Parch'], axis=1)

X_train = train_df.drop("Survived", axis=1)
Y_train = train_df["Survived"]

# 3. Train Random Forest Model with Hyperparameters[cite: 132]
random_forest = RandomForestClassifier(criterion="gini", min_samples_leaf=1, 
                                     min_samples_split=10, n_estimators=100, 
                                     oob_score=True, random_state=1, n_jobs=-1)
random_forest.fit(X_train, Y_train)

print(f"OOB Score: {round(random_forest.oob_score_, 4) * 100}%")

# 4. Model Evaluation[cite: 133, 134, 137]
predictions = cross_val_predict(random_forest, X_train, Y_train, cv=3)

print("Confusion Matrix:\n", confusion_matrix(Y_train, predictions))
print(f"Precision: {precision_score(Y_train, predictions):.4f}")#[cite: 134]
print(f"Recall: {recall_score(Y_train, predictions):.4f}")#[cite: 134]
print(f"F1-Score: {f1_score(Y_train, predictions):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(Y_train, random_forest.predict_proba(X_train)[:, 1]):.4f}")#[cite: 137]

<>:57: SyntaxWarning: invalid escape sequence '\.'
<>:57: SyntaxWarning: invalid escape sequence '\.'
C:\Users\Anushka\AppData\Local\Temp\ipykernel_8424\3096190805.py:57: SyntaxWarning: invalid escape sequence '\.'
  dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)


OOB Score: 82.6%
Confusion Matrix:
 [[490  59]
 [ 90 252]]
Precision: 0.8103
Recall: 0.7368
F1-Score: 0.7718
ROC-AUC Score: 0.9441
